# 02_Wide_Transformations — groupBy/agg, orderBy, join, distinct, repartition, coalesce

**Created:** 2026-02-05 04:59:58

> This notebook teaches PySpark step‑by‑step using the tiny array `123456781`. Everything is explained like you're 5: small stories, simple words, and prints everywhere so you can *see* what happens.


## 0) Spark Setup (ELI5)

- Think of Spark like a **kitchen**. The **driver** is the chef that reads the recipe. The **executors** are helpers who actually chop, stir, and cook.
- We ask the chef to **plan** the recipe (lazy *transformations*), and we only **cook** when we shout "Serve!" (an *action*).

Run the next cell to make a tiny kitchen on your laptop (`local[*]`). If PySpark isn't installed, the cell will tell you how to install it.


In [ ]:

# SparkSession is the doorway to Spark SQL/DataFrames
try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql import types as T
    from pyspark.sql.window import Window
    spark = (SparkSession
             .builder
             .appName("ELI5-PySpark-Operators")
             .master("local[*]")  # use all local cores
             .getOrCreate())
    print("✅ Spark is ready:", spark.version)
except Exception as e:
    print("❌ PySpark not available in this environment.")
    print("Install with: pip install pyspark --quiet")
    raise



## 1) Our tiny toys (data)

We turn `123456781` into:
- An **RDD** of digits (low-level Spark collection), and
- A **DataFrame** with one column `x` (higher-level table‑like structure).

We also add a tiny **index** column to make sorting/join demos easier.


In [ ]:

# Our raw Python list and string
py_list = [1,2,3,4,5,6,7,8,1]
py_str  = "123456781"

# Create an RDD from the list
rdd = spark.sparkContext.parallelize(py_list, 2)  # 2 partitions so you can see partition behavior
print("RDD partitions:", rdd.getNumPartitions())
print("RDD sample:", rdd.take(5))

# Create a DataFrame with an index
df = spark.createDataFrame([(i, v) for i, v in enumerate(py_list)], schema=["idx","x"]) 
print("DataFrame schema:")
df.printSchema()
print("First rows:")
df.show(5, truncate=False)



## 3) Wide Transformations (shuffle happens)
**ELI5:** Helpers must **swap plates** across the room so all lemons go to one table, all apples to another. Swapping = **shuffle** (network heavy).

We'll cover:
- `groupBy().agg()` — group and summarize
- `orderBy()` / `sort()` — sort rows
- `join()` — glue two DataFrames by a key
- `distinct()` — drop duplicates
- `repartition()` — change number of partitions (full shuffle)
- `coalesce()` — shrink partitions (no full shuffle)


In [ ]:

from pyspark.sql import functions as F

print("Original:")
df.show()

# distinct: remove duplicate values
print("Distinct x values:")
dist = df.select("x").distinct()
dist.show()

# groupBy/agg: count and average per value of x%2 (0 even, 1 odd)
print("Group by parity (x % 2): count and avg(x)")
by_parity = (df
    .withColumn("parity", F.col("x") % 2)
    .groupBy("parity")
    .agg(F.count("*").alias("cnt"), F.avg("x").alias("avg_x")))
by_parity.show()

# orderBy: sort by column
print("Order by x descending:")
df.orderBy(F.col("x").desc()).show()

# Build a tiny lookup table to join
lookup = spark.createDataFrame([(1, "one"),(2,"two"),(3,"three"),(4,"four"),(5,"five"),(6,"six"),(7,"seven"),(8,"eight")], ["x","word"]) 
print("Lookup table:")
lookup.show()

print("Inner join on x -> adds the word if present")
joined = df.join(lookup, on="x", how="inner")
joined.show()

# repartition/coalesce demo
print("
Before repartition, df partitions:", df.rdd.getNumPartitions())
rep = df.repartition(4, "x")  # full shuffle, hash on 'x'
print("After repartition(4, 'x'):", rep.rdd.getNumPartitions())
coa = rep.coalesce(2)          # shrink without full shuffle
print("After coalesce(2):", coa.rdd.getNumPartitions())



### Mini‑labs
1) Do a **left** join so all rows from `df` remain even if no word is found.
2) Sort by `x` then by `idx` descending (multi‑column sort).
3) Build `groupBy('x').agg(F.collect_list('idx'))` to see which positions produced each `x`.
